In [1]:
from rank_bm25 import BM25Okapi

print("BM25 imported successfully")

BM25 imported successfully


In [2]:
documents = [
    "Machine learning allows computers to learn patterns from data.",
    "Artificial intelligence enables machines to perform tasks that normally require human intelligence.",
    "Natural language processing helps computers understand and process human language.",
    "Reinforcement learning allows an agent to learn by interacting with an environment.",
    "Statistics helps us analyse data and understand uncertainty."
]

In [3]:
tokenized_documents = [
    document.lower().split()
    for document in documents
]

for document in tokenized_documents:
    print(document)

['machine', 'learning', 'allows', 'computers', 'to', 'learn', 'patterns', 'from', 'data.']
['artificial', 'intelligence', 'enables', 'machines', 'to', 'perform', 'tasks', 'that', 'normally', 'require', 'human', 'intelligence.']
['natural', 'language', 'processing', 'helps', 'computers', 'understand', 'and', 'process', 'human', 'language.']
['reinforcement', 'learning', 'allows', 'an', 'agent', 'to', 'learn', 'by', 'interacting', 'with', 'an', 'environment.']
['statistics', 'helps', 'us', 'analyse', 'data', 'and', 'understand', 'uncertainty.']


In [4]:
bm25 = BM25Okapi(tokenized_documents)

In [5]:
query = "How do computers learn from data?"
tokenized_query = query.lower().split()

print(tokenized_query)

['how', 'do', 'computers', 'learn', 'from', 'data?']


In [6]:
scores = bm25.get_scores(tokenized_query)

print(scores)

[1.87058789 0.         0.33946754 0.31171815 0.        ]


In [8]:
ranked_indices = scores.argsort()[::-1]

for rank, index in enumerate(ranked_indices, start=1):
    print(f"Rank {rank}")
    print(f"Score: {scores[index]:.4f}")
    print(f"Document: {documents[index]}")
    print()

Rank 1
Score: 1.8706
Document: Machine learning allows computers to learn patterns from data.

Rank 2
Score: 0.3395
Document: Natural language processing helps computers understand and process human language.

Rank 3
Score: 0.3117
Document: Reinforcement learning allows an agent to learn by interacting with an environment.

Rank 4
Score: 0.0000
Document: Statistics helps us analyse data and understand uncertainty.

Rank 5
Score: 0.0000
Document: Artificial intelligence enables machines to perform tasks that normally require human intelligence.



In [9]:
def bm25_search(query, documents, bm25, top_k=3):

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    ranked_indices = scores.argsort()[::-1]

    results = []

    for index in ranked_indices[:top_k]:
        results.append({
            "document": documents[index],
            "score": scores[index]
        })

    return results

In [ ]:
results = bm25_search(
    "What allows computers to gain knowledge from examples?",
    documents,
    bm25,
    top_k=3
)

for result in results:
    print(f"{result['score']:.4f} | {result['document']}")


2.1109 | Machine learning allows computers to learn patterns from data.
0.5226 | Reinforcement learning allows an agent to learn by interacting with an environment.
0.3395 | Natural language processing helps computers understand and process human language.


In [11]:
queries = [
    "How do computers learn from data?",
    "How does an agent learn?",
    "What helps computers understand human language?",
    "How do we analyse uncertainty?"
]

for query in queries:

    print(f"\nQuery: {query}")
    print("-" * 60)

    results = bm25_search(
        query,
        documents,
        bm25,
        top_k=2
    )

    for result in results:
        print(f"{result['score']:.4f} | {result['document']}")


Query: How do computers learn from data?
------------------------------------------------------------
1.8706 | Machine learning allows computers to learn patterns from data.
0.3395 | Natural language processing helps computers understand and process human language.

Query: How does an agent learn?
------------------------------------------------------------
2.5030 | Reinforcement learning allows an agent to learn by interacting with an environment.
0.0000 | Statistics helps us analyse data and understand uncertainty.

Query: What helps computers understand human language?
------------------------------------------------------------
1.3579 | Natural language processing helps computers understand and process human language.
0.7453 | Statistics helps us analyse data and understand uncertainty.

Query: How do we analyse uncertainty?
------------------------------------------------------------
1.2167 | Statistics helps us analyse data and understand uncertainty.
0.0000 | Reinforcement lear

In [12]:
evaluation_data = [
    {
        "query": "How do computers learn from data?",
        "relevant_document": 0
    },
    {
        "query": "How does an agent learn?",
        "relevant_document": 3
    },
    {
        "query": "What helps computers understand human language?",
        "relevant_document": 2
    },
    {
        "query": "How do we analyse uncertainty?",
        "relevant_document": 4
    }
]

In [13]:
def bm25_hit_at_k(query, relevant_document, k):

    results = bm25_search(
        query,
        documents,
        bm25,
        top_k=k
    )

    retrieved_documents = []

    for result in results:
        retrieved_documents.append(result["document"])

    relevant_text = documents[relevant_document]

    return relevant_text in retrieved_documents

In [14]:
result = bm25_hit_at_k(
    "How do computers learn from data?",
    0,
    2
)

print(result)

True


In [15]:
for item in evaluation_data:

    result = bm25_hit_at_k(
        item["query"],
        item["relevant_document"],
        k=2
    )

    print(f"Query: {item['query']}")
    print(f"Hit@2: {result}")
    print()

Query: How do computers learn from data?
Hit@2: True

Query: How does an agent learn?
Hit@2: True

Query: What helps computers understand human language?
Hit@2: True

Query: How do we analyse uncertainty?
Hit@2: True



In [16]:
hits = 0

for item in evaluation_data:

    result = bm25_hit_at_k(
        item["query"],
        item["relevant_document"],
        k=2
    )

    if result:
        hits += 1

hit_at_2 = hits / len(evaluation_data)

print(f"BM25 Hit@2: {hit_at_2:.2f}")

BM25 Hit@2: 1.00


| TF-IDF                                            | BM25                                                                                  |
| ------------------------------------------------- | ------------------------------------------------------------------------------------- |
| Basic classical retrieval method                  | More advanced classical retrieval method                                              |
| Uses term frequency + inverse document frequency  | Uses term frequency, inverse document frequency, document length, and term saturation |
| Uses cosine similarity in our implementation      | Uses BM25 relevance score                                                             |
| Can be affected by repeated words/document length | Handles repeated terms and document length more carefully                             |
| Simple and useful baseline                        | Usually a stronger classical retrieval approach                                       |


## Conclusion

BM25 provides a more advanced lexical retrieval method than TF-IDF.
Both methods were tested using the same queries and relevance labels.

In this small experiment, BM25 achieved a Hit@2 of 1.00.
However, the dataset is very small, so this result cannot be used
to conclude that BM25 is generally better than TF-IDF.

The next step is to explore semantic search using embeddings,
which can capture meaning beyond exact word matching.